# M04 — Historical-Simulation VaR and Expected Shortfall

This notebook reproduces M04 from a fresh Google Colab runtime. It applies complete observed one-day GSW zero-curve changes to the current frozen 3Y/7Y/15Y synthetic Treasury portfolio, performs full repricing, and estimates VaR and Expected Shortfall. It does not backtest forecasts or implement stress testing.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JoyWu-302121/market_risk.git'
PROJECT_DIR = Path('/content/market_risk') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB and not (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
elif IN_COLAB:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print(f'Project ready at {PROJECT_DIR}')

## Load configuration and current public data

The processed history begins at the accepted 2005-01-01 analysis boundary. The valuation curve is the latest date with every configured 1Y-30Y node.

In [ ]:
import pandas as pd
import yaml
from IPython.display import display

from bond_risk.curves import ZeroCurve, latest_complete_curve_date
from bond_risk.data.pipeline import run_gsw_pipeline
from bond_risk.portfolio import build_target_weight_portfolio

data_configuration = yaml.safe_load((PROJECT_DIR / 'configs/data_sources.yaml').read_text())['gsw']
portfolio_configuration = yaml.safe_load((PROJECT_DIR / 'configs/portfolio.yaml').read_text())
risk_configuration = yaml.safe_load((PROJECT_DIR / 'configs/risk.yaml').read_text())['historical_simulation']
OUTPUT_ROOT = PROJECT_DIR / 'data'
ingestion = run_gsw_pipeline(
    OUTPUT_ROOT,
    analysis_start=data_configuration['analysis_start'],
    required_tenors=data_configuration['required_tenors_years'],
    source_url=data_configuration['url'],
    stale_after_days=data_configuration['stale_after_calendar_days'],
)
assert ingestion['audit']['status'] != 'FAIL', ingestion['audit']['failures']
curve_data = pd.read_csv(ingestion['processed_path'], parse_dates=['observation_date'])
curve_data = curve_data.loc[curve_data['observation_date'] >= pd.Timestamp(data_configuration['analysis_start'])].copy()
required_tenors = [float(value) for value in risk_configuration['required_curve_tenors_years']]
valuation_date = latest_complete_curve_date(curve_data, required_tenors=required_tenors)
base_curve = ZeroCurve.from_long_frame(curve_data, observation_date=valuation_date)
portfolio = build_target_weight_portfolio(
    portfolio_id=portfolio_configuration['portfolio_id'],
    curve=base_curve,
    total_market_value=portfolio_configuration['initial_market_value'],
    instrument_specs=portfolio_configuration['positions'],
)
print(f'Valuation date: {valuation_date.date()}')
display(portfolio.position_report(base_curve))

## Build and fully reprice historical scenarios

Only adjacent dates with all required observed nodes create a successful shock. Failed transitions remain visible in the audit table and are never converted to zero losses.

In [ ]:
from bond_risk.scenarios import build_historical_curve_shocks, revalue_historical_scenarios

shock_set = build_historical_curve_shocks(
    curve_data,
    required_tenors=required_tenors,
    as_of_date=valuation_date,
)
scenario_results = revalue_historical_scenarios(
    base_curve=base_curve,
    portfolio=portfolio,
    shock_set=shock_set,
)
display(scenario_results['status'].value_counts().rename('scenario_count').to_frame())
display(scenario_results.tail())

In [ ]:
import matplotlib.pyplot as plt

successful = scenario_results.loc[scenario_results['status'] == 'success'].copy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(successful['loss'] / 1_000, bins=60, color='#2f6f9f', alpha=0.85)
ax.set(title='Historical Full-Revaluation Loss Distribution', xlabel='One-day loss (USD thousands)', ylabel='Scenario count')
ax.axvline(0, color='black', linewidth=1)
ax.grid(axis='y', alpha=0.25)
plt.show()

## Estimate rolling-window VaR and ES

VaR uses the nearest-rank order statistic. ES averages the worst `n × (1 - alpha)` observations with fractional weight on the boundary loss when required.

In [ ]:
from bond_risk.risk import empirical_var, estimate_historical_risk

estimates = estimate_historical_risk(
    scenario_results,
    valuation_date=valuation_date.date().isoformat(),
    windows=risk_configuration['estimation_windows'],
    var_confidence_levels=risk_configuration['var_confidence_levels'],
    es_confidence_levels=risk_configuration['es_confidence_levels'],
    horizon_days=risk_configuration['horizon_days'],
)
risk_table = estimates.pivot(index='window_size', columns=['measure', 'confidence_level'], values='value')
display(risk_table.style.format('${:,.2f}'))

## Verify M04 invariants

In [ ]:
import json
import numpy as np

assert len(base_curve.tenors_years) == 30
assert len(successful) >= max(risk_configuration['estimation_windows'])
assert pd.to_datetime(scenario_results['shock_end_date']).max() <= valuation_date
position_columns = [column for column in successful if column.startswith('position_loss__')]
np.testing.assert_allclose(successful[position_columns].sum(axis=1), successful['loss'], atol=1e-8)
for row in estimates.itertuples(index=False):
    assert np.isclose(sum(json.loads(row.position_contributions).values()), row.value, atol=1e-8)
for window in risk_configuration['estimation_windows']:
    sample = successful.sort_values('shock_end_date').tail(window)
    for alpha in risk_configuration['es_confidence_levels']:
        es_value = estimates.loc[(estimates['window_size'] == window) & (estimates['measure'] == 'ES') & (estimates['confidence_level'] == alpha), 'value'].iloc[0]
        assert es_value >= empirical_var(sample['loss'], alpha) - 1e-8
print('M04 historical VaR/ES acceptance checks: PASS')

## M04 completion boundary

M04 estimates current historical-simulation risk for three rolling lookback windows. M05 will add deterministic, historical-event, PCA-factor, and reverse stress tests. M06 will perform out-of-sample VaR and ES validation.